1. Install & Imports

In [1]:
!pip install -q transformers scikit-learn

import os
import zipfile
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import CLIPProcessor, CLIPModel

2. Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


3. Paths

In [3]:
BASE_PATH = '/content/drive/MyDrive/memotion_project'
ZIP_PATH = os.path.join(BASE_PATH, 'dataset.zip')
EXTRACT_PATH = os.path.join(BASE_PATH, 'memotion_raw')

os.makedirs(EXTRACT_PATH, exist_ok=True)

4. Extract Dataset

In [4]:
if not os.listdir(EXTRACT_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

5. Load CSV

In [5]:
CSV_PATH = None

for root, dirs, files in os.walk(EXTRACT_PATH):
    for f in files:
        if f == 'labels.csv':
            CSV_PATH = os.path.join(root, f)

df = pd.read_csv(CSV_PATH)

print(df.columns)

# ✅ DEFINE TEXT COLUMN HERE
TEXT_COL = 'text_ocr'

Index(['Unnamed: 0', 'image_name', 'text_ocr', 'text_corrected', 'humour',
       'sarcasm', 'offensive', 'motivational', 'overall_sentiment'],
      dtype='object')


In [6]:
print(TEXT_COL)

text_ocr


6. Label Setup (MULTI-LABEL)

In [7]:
from sklearn.preprocessing import LabelEncoder

LABEL_COLUMNS = ['humour', 'sarcasm', 'offensive', 'motivational']

encoders = {}

for col in LABEL_COLUMNS:
    # Convert everything to string first
    df[col] = df[col].astype(str)

    # Handle missing values
    df[col] = df[col].fillna("unknown")

    # Encode
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

    encoders[col] = le

print("Label encoding done")
print(df[LABEL_COLUMNS].head())

Label encoding done
   humour  sarcasm  offensive  motivational
0       1        0          1             1
1       2        0          1             0
2       3        1          1             1
3       3        2          3             0
4       1        3          3             1


In [8]:
print(df[LABEL_COLUMNS].dtypes)

humour          int64
sarcasm         int64
offensive       int64
motivational    int64
dtype: object


7. Train  Val Split

In [9]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

8. Load CLIP

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
clip_model = CLIPModel.from_pretrained(model_name)
clip_model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

9. Dataset Class

In [11]:
def load_image(path):
    try:
        return Image.open(path).convert("RGB")
    except:
        return None

class MemeDataset(torch.utils.data.Dataset):
    def __init__(self, df, text_col):
        self.df = df
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        IMG_DIR = os.path.join(os.path.dirname(CSV_PATH), 'images')


        base_name = os.path.splitext(row['image_name'])[0]

        # try all possible extensions
        found = False
        for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
            img_path = os.path.join(IMG_DIR, base_name + ext)
            if os.path.exists(img_path):
                found = True
                break

        if not found:
            return None
        text = str(row[self.text_col]) if pd.notna(row[self.text_col]) else ""
        labels = torch.tensor(row[LABEL_COLUMNS].values.astype(float), dtype=torch.float)

        image = load_image(img_path)
        if image is None:
            return None

        inputs = processor(
            text=[text],
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77
        )

        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item['labels'] = labels
        return item

10. Collate Function

In [12]:
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    return {
        "input_ids":      torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "pixel_values":   torch.stack([b["pixel_values"].squeeze(0) for b in batch]),
        "labels":         torch.stack([b["labels"] for b in batch])
    }

11. Dataloaders

In [13]:
train_loader = torch.utils.data.DataLoader(
    MemeDataset(train_df, TEXT_COL),
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = torch.utils.data.DataLoader(
    MemeDataset(val_df, TEXT_COL),
    batch_size=16,
    collate_fn=collate_fn
)

Multimodal Model

In [21]:
class MultimodalModel(nn.Module):
    def __init__(self, clip_model, num_labels):
        super().__init__()
        self.clip = clip_model

        # auto-detect actual embedding sizes
        vision_dim = clip_model.config.vision_config.hidden_size   # e.g. 768 or 1280
        text_dim   = clip_model.config.text_config.hidden_size     # e.g. 512 or 1280
        fused_dim  = vision_dim + text_dim

        print(f"vision_dim={vision_dim}, text_dim={text_dim}, fused_dim={fused_dim}")

        self.fc = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_labels)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        image_features = self.clip.vision_model(pixel_values=pixel_values).pooler_output
        text_features  = self.clip.text_model(input_ids=input_ids, attention_mask=attention_mask).pooler_output

        fused  = torch.cat((image_features, text_features), dim=1)
        logits = self.fc(fused)
        return logits

13. Initialize Model

In [22]:
model = MultimodalModel(clip_model, len(LABEL_COLUMNS)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

vision_dim=768, text_dim=512, fused_dim=1280


14. Training Loop

In [16]:
# FIXED train_epoch: single loop, correct count
def train_epoch(loader):
    model.train()
    total_loss = 0
    count = 0                      # count here, not a second loop

    for batch in tqdm(loader):
        if batch is None:
            continue

        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(batch['input_ids'], batch['attention_mask'], batch['pixel_values'])
        loss = criterion(logits, batch['labels'])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1                  # increment in same loop

    return total_loss / max(count, 1)

15. Validation Loop

In [24]:
# FIXED evaluate: use threshold=0.5 for multi-label, not argmax
def evaluate(loader):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch['input_ids'], batch['attention_mask'], batch['pixel_values'])
            pred = torch.argmax(logits, dim=1).cpu()
            target = torch.argmax(batch['labels'], dim=1).cpu()

            preds.append(pred)
            targets.append(target)

    if not preds:
        print("No valid validation samples!")
        return 0, 0

    preds   = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()

    acc = accuracy_score(targets, preds)
    f1  = f1_score(targets, preds, average='macro')
    return acc, f1

In [18]:
valid = 0
for i in range(100):
    if MemeDataset(df, TEXT_COL)[i] is not None:
        valid += 1

print("Valid samples:", valid)

Valid samples: 99


In [19]:
for batch in train_loader:
    print(batch['pixel_values'].shape)
    break

torch.Size([16, 3, 224, 224])


16. Train Model

In [25]:
EPOCHS = 3

for epoch in range(EPOCHS):
    loss = train_epoch(train_loader)
    acc, f1 = evaluate(val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")

  1%|          | 2/350 [00:01<03:45,  1.54it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  7%|▋         | 26/350 [00:14<03:00,  1.80it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 350/350 [03:16<00:00,  1.78it/s]


Epoch 1
Loss: -25.5201 | Acc: 0.3269 | F1: 0.1232


 47%|████▋     | 166/350 [01:33<01:44,  1.77it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 81%|████████▏ | 285/350 [02:40<00:37,  1.76it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 350/350 [03:17<00:00,  1.77it/s]


Epoch 2
Loss: -62.2526 | Acc: 0.3269 | F1: 0.1232


  5%|▍         | 17/350 [00:09<03:03,  1.82it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  9%|▉         | 31/350 [00:17<03:01,  1.76it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 350/350 [03:20<00:00,  1.75it/s]


Epoch 3
Loss: -116.8542 | Acc: 0.3269 | F1: 0.1232


17. Save Model

In [26]:
torch.save(model.state_dict(), BASE_PATH + "/multimodal_model.pth")

In [27]:
torch.save(model.state_dict(), "/content/model.pth")

from google.colab import files
files.download("/content/model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>